<a href="https://colab.research.google.com/github/PranjanaNarayan/-ethical-ai-bias-detector-/blob/main/ethical_ai_bias_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fairlearn aif360 -q
print("All libraries ready!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 5000

# Generate candidate data
age = np.random.randint(22, 60, n)
experience = np.random.randint(0, 20, n)
education = np.random.choice([0, 1, 2], n)
test_score = np.random.normal(70, 15, n).clip(0, 100)
gender = np.random.choice([0, 1], n)
race = np.random.choice([0, 1, 2, 3], n)

# Introduce BIAS — unfairly penalize women and minorities
hire_prob = (
    0.3 * (test_score / 100) +
    0.2 * (experience / 20) +
    0.1 * education +
    0.2 * (gender == 1) +      # bias against women
    0.1 * (race == 0) +        # bias against minorities
    np.random.normal(0, 0.1, n)
)

hired = (hire_prob > 0.5).astype(int)

df = pd.DataFrame({
    'age': age,
    'experience': experience,
    'education': education,
    'test_score': test_score.round(2),
    'gender': gender,
    'race': race,
    'hired': hired
})

print("Dataset created!")
print("Shape:", df.shape)
print("\nHiring rate by gender:")
print(df.groupby('gender')['hired'].mean().round(3))
print("\nHiring rate by race:")
print(df.groupby('race')['hired'].mean().round(3))
print("\n0=Male 1=Female | 0=Minority groups 1,2,3=Majority")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bias Analysis in Hiring Data', fontsize=16)

# Plot 1 - Hiring rate by gender
gender_hire = df.groupby('gender')['hired'].mean()
axes[0][0].bar(['Male', 'Female'], gender_hire.values,
               color=['#2196F3', '#E91E63'])
axes[0][0].set_title('Hiring Rate by Gender')
axes[0][0].set_ylabel('Hiring Rate')
axes[0][0].set_ylim(0, 1)
for i, v in enumerate(gender_hire.values):
    axes[0][0].text(i, v+0.02, f'{v:.2%}', ha='center')

# Plot 2 - Hiring rate by race
race_hire = df.groupby('race')['hired'].mean()
axes[0][1].bar(['Group A', 'Group B', 'Group C', 'Group D'],
               race_hire.values,
               color=['#FF5722', '#4CAF50', '#9C27B0', '#FF9800'])
axes[0][1].set_title('Hiring Rate by Race')
axes[0][1].set_ylabel('Hiring Rate')
axes[0][1].set_ylim(0, 1)

# Plot 3 - Test score vs hired by gender
for g, label, color in [(0,'Male','#2196F3'),
                         (1,'Female','#E91E63')]:
    data = df[df['gender']==g]
    axes[1][0].scatter(data['test_score'],
                       data['experience'],
                       c=data['hired'],
                       alpha=0.3, label=label)
axes[1][0].set_title('Test Score vs Experience (colored by hired)')
axes[1][0].set_xlabel('Test Score')
axes[1][0].set_ylabel('Experience')

# Plot 4 - Correlation heatmap
axes[1][1].set_title('Feature Correlation with Hiring')
corr = df.corr()['hired'].drop('hired').sort_values()
colors_bar = ['red' if x < 0 else 'green' for x in corr.values]
axes[1][1].barh(corr.index, corr.values, color=colors_bar)
axes[1][1].axvline(x=0, color='black', linestyle='-')
axes[1][1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

print("\nKey Bias Finding:")
male_rate = df[df['gender']==0]['hired'].mean()
female_rate = df[df['gender']==1]['hired'].mean()
print(f"Male hiring rate:   {male_rate:.2%}")
print(f"Female hiring rate: {female_rate:.2%}")
print(f"Gender bias gap:    {(male_rate-female_rate):.2%}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

features = ['age', 'experience', 'education', 'test_score',
            'gender', 'race']
X = df[features]
y = df['hired']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training biased model...")
biased_model = RandomForestClassifier(
    n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
y_pred_biased = biased_model.predict(X_test)

print("Biased Model Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_biased):.4f}")
print(classification_report(y_test, y_pred_biased)

In [ ]:
from fairlearn.metrics import (MetricFrame,
                                selection_rate,
                                demographic_parity_difference,
                                equalized_odds_difference)

# Analyze bias by gender
gender_test = X_test['gender']

metric_frame = MetricFrame(
    metrics={
        'selection_rate': selection_rate,
        'accuracy': accuracy_score
    },
    y_true=y_test,
    y_pred=y_pred_biased,
    sensitive_features=gender_test
)

print("="*50)
print("BIAS DETECTION REPORT")
print("="*50)
print("\nMetrics by Gender:")
print(metric_frame.by_group)

dpd = demographic_parity_difference(
    y_test, y_pred_biased,
    sensitive_features=gender_test)

eod = equalized_odds_difference(
    y_test, y_pred_biased,
    sensitive_features=gender_test)

print(f"\nDemographic Parity Difference: {dpd:.4f}")
print(f"(0 = perfectly fair, closer to 0 = less bias)")
print(f"\nEqualized Odds Difference: {eod:.4f}")

if abs(dpd) > 0.1:
    print("\n  SIGNIFICANT BIAS DETECTED!")
    print("This model discriminates based on gender!")
else:
    print("\n Model is relatively fair")

In [ ]:
# Remove sensitive features to reduce bias
fair_features = ['age', 'experience', 'education', 'test_score']
X_train_fair = X_train[fair_features]
X_test_fair = X_test[fair_features]

print("Training fair model (without sensitive features)...")
fair_model = RandomForestClassifier(
    n_estimators=100, random_state=42)
fair_model.fit(X_train_fair, y_train)
y_pred_fair = fair_model.predict(X_test_fair)

print("Fair Model Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_fair):.4f}")

# Check bias again
metric_frame_fair = MetricFrame(
    metrics={
        'selection_rate': selection_rate,
        'accuracy': accuracy_score
    },
    y_true=y_test,
    y_pred=y_pred_fair,
    sensitive_features=gender_test
)

dpd_fair = demographic_parity_difference(
    y_test, y_pred_fair,
    sensitive_features=gender_test)

print("\nMetrics by Gender (Fair Model):")
print(metric_frame_fair.by_group)
print(f"\nDemographic Parity Difference: {dpd_fair:.4f}")
print(f"\nBias Reduction: {abs(dpd)-abs(dpd_fair):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Bias Detection and Mitigation Results', fontsize=14)

# Biased model
biased_rates = metric_frame.by_group['selection_rate']
axes[0].bar(['Male', 'Female'], biased_rates.values,
            color=['#2196F3', '#E91E63'])
axes[0].set_title(f'Biased Model\nDPD = {dpd:.4f} ⚠️')
axes[0].set_ylabel('Selection Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(biased_rates.values):
    axes[0].text(i, v+0.02, f'{v:.2%}', ha='center')

# Fair model
fair_rates = metric_frame_fair.by_group['selection_rate']
axes[1].bar(['Male', 'Female'], fair_rates.values,
            color=['#2196F3', '#E91E63'])
axes[1].set_title(f'Fair Model\nDPD = {dpd_fair:.4f} ✅')
axes[1].set_ylabel('Selection Rate')
axes[1].set_ylim(0, 1)
for i, v in enumerate(fair_rates.values):
    axes[1].text(i, v+0.02, f'{v:.2%}', ha='center')

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Biased Model Accuracy:  {accuracy_score(y_test, y_pred_biased):.4f}")
print(f"Fair Model Accuracy:    {accuracy_score(y_test, y_pred_fair):.4f}")
print(f"\nBiased Model DPD:  {dpd:.4f}")
print(f"Fair Model DPD:    {dpd_fair:.4f}")
print(f"\nBias reduced by:   {abs(dpd)-abs(dpd_fair):.4f}")
print("\nConclusion: Removing sensitive features")
print("reduces bias while maintaining accuracy!")

In [ ]:
import pickle
from google.colab import files

with open('bias_detector.pkl', 'wb') as f:
    pickle.dump({
        'biased_model': biased_model,
        'fair_model': fair_model,
        'biased_dpd': dpd,
        'fair_dpd': dpd_fair
    }, f)

print("Model saved!")
files.download('bias_detector.pkl')